In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

import tensorflow_model_optimization as tfmot

In [ ]:
dataset_path = "../dataset"

classes = [
    "benign",
    "constant",
    "random",
    "reactive"
]

In [ ]:
dataframes = []

for cls in classes:

    folder = os.path.join(dataset_path, cls)

    for file in os.listdir(folder):

        path = os.path.join(folder, file)

        df = pd.read_csv(path)

        df["label"] = cls

        dataframes.append(df)

dataset = pd.concat(dataframes, ignore_index=True)

print("Dataset shape:", dataset.shape)

dataset.head()

In [ ]:
dataset.isnull().sum()

In [ ]:
plt.figure()

dataset["label"].value_counts().plot(kind="bar")

plt.title("Class Distribution")

plt.show()

In [ ]:
selected_features = [
    "snr",
    "rssi",
    "signal_power",
    "noise_power",
    "packet_loss",
    "throughput",
    "latency",
    "jitter",
    "modulation_index",
    "frequency_offset",
    "bandwidth",
    "spectral_entropy"
]

X = dataset[selected_features]

y = dataset["label"]

print(X.shape)

In [ ]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("Classes:", encoder.classes_)

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)

In [ ]:
model = Sequential()

model.add(Dense(32, activation="relu", input_shape=(12,)))
model.add(Dropout(0.2))

model.add(Dense(16, activation="relu"))

model.add(Dense(4, activation="softmax"))

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])

plt.title("Training Accuracy")

plt.legend(["train","validation"])

plt.show()

In [ ]:
predictions = model.predict(X_test)

y_pred = np.argmax(predictions, axis=1)

acc = accuracy_score(y_test, y_pred)

print("Test Accuracy:", acc)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True)

plt.title("Confusion Matrix")

plt.show()

In [ ]:
target_names = [
    "benign",
    "constant_jammer",
    "random_jammer",
    "reactive_jammer"
]

print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_names
    )
)

In [ ]:
pruning_params = {
    "pruning_schedule":
    tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=2000
    )
}

model_pruned = tfmot.sparsity.keras.prune_low_magnitude(
    model,
    **pruning_params
)

In [ ]:
model_pruned.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model_pruned.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64
)

In [ ]:
model_pruned = tfmot.sparsity.keras.strip_pruning(model_pruned)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_pruned)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("../models/model_quantized.tflite","wb") as f:
    f.write(tflite_model)

print("Quantized model saved")

In [ ]:
model_pruned.save("../models/jamshield_model_pruned.h5")

print("Model saved")

In [ ]:
weights = model_pruned.get_weights()

np.save("../models/fpga_weights.npy", weights)

print("Weights exported for FPGA")

In [ ]:
sample = X_test[0]

print(sample)